# Merval Financial Analysis — ETL Pipeline

In this notebook we transform the raw price data and calculate 
key financial metrics for analysis.

## Objectives
- Calculate daily returns
- Calculate cumulative returns
- Calculate volatility (rolling std)
- Calculate moving averages (MA20 / MA50)
- Identify unusual volume spikes
- Export clean data to SQL Server

In [1]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv

# Cargar API key desde .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("ALPHA_VANTAGE_KEY")

# Acciones a analizar
tickers = {
    "YPF": "YPF",
    "Galicia": "GGAL",
    "MercadoLibre": "MELI",
    "Tenaris": "TS",
    "Pampa": "PAM"
}

def get_stock_data(symbol, api_key):
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&apikey={api_key}"
    response = requests.get(url)
    data = response.json()
    
    if 'Time Series (Daily)' not in data:
        print(f"Error en {symbol}: {data}")
        return None
    
    time_series = data['Time Series (Daily)']
    df = pd.DataFrame(time_series).T
    df.index = pd.to_datetime(df.index)
    df.columns = ['open', 'high', 'low', 'close', 'volume']
    df = df.astype(float)
    df['symbol'] = symbol
    return df

dfs = []
for nombre, ticker in tickers.items():
    print(f"Descargando {nombre} ({ticker})...")
    df = get_stock_data(ticker, API_KEY)
    if df is not None:
        dfs.append(df)
    time.sleep(15)

df_all = pd.concat(dfs)
df_all = df_all.reset_index().rename(columns={'index': 'date'})

print(f"\nDatos cargados: {df_all.shape}")

Descargando YPF (YPF)...
Descargando Galicia (GGAL)...
Descargando MercadoLibre (MELI)...
Descargando Tenaris (TS)...
Descargando Pampa (PAM)...

Datos cargados: (500, 7)


In [2]:
# Ordenar por símbolo y fecha
df_all = df_all.sort_values(['symbol', 'date']).reset_index(drop=True)

# Calcular métricas por acción
def calculate_metrics(df):
    df = df.copy()
    
    # Retorno diario
    df['daily_return'] = df['close'].pct_change() * 100
    
    # Retorno acumulado
    df['cumulative_return'] = ((df['close'] / df['close'].iloc[0]) - 1) * 100
    
    # Volatilidad rolling 20 días
    df['volatility_20d'] = df['daily_return'].rolling(20).std()
    
    # Medias móviles
    df['ma20'] = df['close'].rolling(20).mean()
    df['ma50'] = df['close'].rolling(50).mean()
    
    # Volumen promedio 20 días
    df['volume_avg_20d'] = df['volume'].rolling(20).mean()
    
    # Spike de volumen (volumen actual vs promedio)
    df['volume_spike'] = df['volume'] / df['volume_avg_20d']
    
    return df

df_metrics = df_all.groupby('symbol', group_keys=False).apply(calculate_metrics)

print(f"Dimensiones: {df_metrics.shape}")
print(f"\nColumnas: {df_metrics.columns.tolist()}")
print(f"\nNulos por columna:")
print(df_metrics.isnull().sum())

Dimensiones: (500, 13)

Columnas: ['date', 'open', 'high', 'low', 'close', 'volume', 'daily_return', 'cumulative_return', 'volatility_20d', 'ma20', 'ma50', 'volume_avg_20d', 'volume_spike']

Nulos por columna:
date                   0
open                   0
high                   0
low                    0
close                  0
volume                 0
daily_return           5
cumulative_return      0
volatility_20d       100
ma20                  95
ma50                 245
volume_avg_20d        95
volume_spike          95
dtype: int64


In [8]:
# Reconstruir df_metrics correctamente
dfs_fixed = []
for symbol, group in df_all.groupby('symbol'):
    temp = calculate_metrics(group.copy())
    temp['symbol'] = symbol
    dfs_fixed.append(temp)

df_metrics = pd.concat(dfs_fixed).reset_index(drop=True)

print(df_metrics['symbol'].unique())
print(df_metrics.shape)

<StringArray>
['GGAL', 'MELI', 'PAM', 'TS', 'YPF']
Length: 5, dtype: str
(500, 14)


In [9]:
# Resumen por acción
resumen = df_metrics.groupby('symbol').agg(
    retorno_acumulado=('cumulative_return', 'last'),
    volatilidad_promedio=('volatility_20d', 'mean'),
    max_volume_spike=('volume_spike', 'max'),
    precio_actual=('close', 'last'),
    ma20_actual=('ma20', 'last'),
    ma50_actual=('ma50', 'last')
).round(2)

print(resumen)

        retorno_acumulado  volatilidad_promedio  max_volume_spike  \
symbol                                                              
GGAL               -15.65                  3.19              1.71   
MELI               -30.55                  2.61              3.80   
PAM                  6.01                  1.99              2.27   
TS                  43.16                  1.85              3.22   
YPF                 23.48                  2.27              3.60   

        precio_actual  ma20_actual  ma50_actual  
symbol                                           
GGAL            45.36        43.61        48.05  
MELI          1639.47      1716.27      1922.25  
PAM             85.85        81.55        82.56  
TS              57.18        54.56        49.85  
YPF             43.18        38.23        37.72  


In [10]:
EXPORT_PATH = "../exports/"
os.makedirs(EXPORT_PATH, exist_ok=True)

df_metrics.to_csv(EXPORT_PATH + "merval_metrics.csv", index=False)
print(f"Exportado: {df_metrics.shape[0]:,} filas → merval_metrics.csv")

Exportado: 500 filas → merval_metrics.csv


In [11]:
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine

# Renombrar columnas para que coincidan con SQL Server
df_sql = df_metrics.rename(columns={
    'date': 'trade_date',
    'open': 'open_price',
    'high': 'high_price',
    'low': 'low_price',
    'close': 'close_price'
})

engine = create_engine(
    "mssql+pyodbc://localhost/merval_analysis?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

df_sql.to_sql('merval_metrics', con=engine, if_exists='replace', index=False)

print(f"Datos cargados exitosamente: {len(df_sql):,} filas")

Datos cargados exitosamente: 500 filas
